# XGBoost Encoding Benchmark – pp_bi

Benchmarks 4 categorical encoding strategies on the pp_bi (Bodily Injury pure premium) target.

| Step | Description |
|------|-------------|
| 0 | Set DEBUG flag – `1` = 10K rows, `0` = full data |
| 1 | (First run only) Generate `config/level_mapping_reference.csv` |
| 2 | Load train / test data |
| 3 | Run 4 encoding experiments + train XGBoost |
| 4 | Compare results |
| 5 | Save all models & artifacts |

**Files used:**
- `code/create_level_mapping.py` – Program 1: builds the mapping CSV  
- `code/encoding_strategies.py` – 4 encoding functions  
- `code/model_training.py`       – Train, evaluate, save, compare  

**Outputs:**
- `models/pp_bi_type{1-4}_*/` – Model + encoders + predictions  
- `results/encoding_benchmark_summary.csv` – Comparison table

---
## 0 · Setup & DEBUG flag

In [ ]:
import os, sys

PROJECT_ROOT = os.path.abspath(os.getcwd())
sys.path.insert(0, os.path.join(PROJECT_ROOT, "code"))

# ── Toggle this flag ─────────────────────────────────────────────────────────
DEBUG = 0   # 1 = 10K train rows (fast test)  |  0 = full dataset
# ─────────────────────────────────────────────────────────────────────────────

print(f"Project root : {PROJECT_ROOT}")
print(f"DEBUG        : {DEBUG}  ({'10K rows' if DEBUG else 'FULL DATA'})")

---
## 1 · Generate level mapping CSV (run once)

Skip this cell if `config/level_mapping_reference.csv` already exists.

In [ ]:
import importlib, create_level_mapping as clm

MAPPING_CSV = os.path.join(PROJECT_ROOT, "config", "level_mapping_reference.csv")

if not os.path.exists(MAPPING_CSV):
    print("Mapping CSV not found – generating now ...")
    # Temporarily override the DEBUG flag inside the module
    clm.DEBUG = DEBUG
    clm.main()
else:
    print(f"✅ Mapping CSV already exists:\n   {MAPPING_CSV}")
    print("   Delete it and re-run this cell to regenerate.")

---
## 2 · Load train and test data

In [ ]:
from encoding_strategies import load_train_test, get_y

train_raw, test_raw = load_train_test(debug=bool(DEBUG))

y_train = get_y(train_raw)
y_test  = get_y(test_raw)

print(f"y_train : mean={y_train.mean():.4f}  std={y_train.std():.4f}")
print(f"y_test  : mean={y_test.mean():.4f}  std={y_test.std():.4f}")

---
## 2.5 · Quick Iterative Experiment

**Iterative tuning workflow:**
1. Edit `config.json` (change `learning_rate`, `n_estimators`, etc.)
2. Run this cell
3. See lift chart + metrics immediately
4. Repeat — no kernel restart needed

Set `ITER_ENC` to choose which encoding to benchmark:
- `1` = Type 1 Ordinal
- `2` = Type 2 Binary
- `3` = Type 3 Actuarial
- `4` = Type 4 Custom

In [ ]:
# ── Edit these two lines, then run ───────────────────────────────────────────
ITER_ENC  = 1          # Encoding type: 1 / 2 / 3 / 4
ITER_NAME = 'run_v1'   # Label for this run (shows in lift chart title)
# ─────────────────────────────────────────────────────────────────────────────

import model_training as _mt
_mt.reload_config()   # Re-read config.json — no kernel restart needed

from encoding_strategies import encode_type1_ordinal, encode_type2_binary
from encoding_strategies import encode_type3_actuarial, encode_type4_custom
from model_training import train_and_evaluate
from visualization import lift_chart

_enc_map = {
    1: lambda: encode_type1_ordinal(train_raw, test_raw) + (None,),
    2: lambda: encode_type2_binary(train_raw,  test_raw) + (None,),
    3: lambda: encode_type3_actuarial(train_raw, test_raw),
    4: lambda: encode_type4_custom(train_raw, test_raw) + (None,),
}
_enc_names = {1:'type1_ordinal', 2:'type2_binary', 3:'type3_actuarial', 4:'type4_custom'}

if ITER_ENC not in _enc_map:
    raise ValueError(f'ITER_ENC must be 1/2/3/4, got {ITER_ENC}')

_out = _enc_map[ITER_ENC]()
_X_tr, _X_te, _feats = _out[0], _out[1], _out[2]

_result = train_and_evaluate(
    experiment_name = f"{_enc_names[ITER_ENC]}_{ITER_NAME}",
    X_train=_X_tr, y_train=y_train,
    X_test =_X_te, y_test =y_test,
    feature_names=_feats,
)

lift_chart(_result, test_df=test_raw, bins=10, print_table=False,
           title=f'Lift — {_enc_names[ITER_ENC]} | {ITER_NAME} | RMSE={_result["metrics"]["rmse"]:.2f}')

print(f'  RMSE : {_result["metrics"]["rmse"]:.4f}')
print(f'  MAE  : {_result["metrics"]["mae"]:.4f}')
print(f'  R²   : {_result["metrics"]["r2"]:.4f}')

---
## 3 · Experiment 1 – Ordinal encoding (Type 1)

In [ ]:
from encoding_strategies import encode_type1_ordinal
from model_training import train_and_evaluate, save_experiment

X_tr1, X_te1, feats1 = encode_type1_ordinal(train_raw, test_raw)

result1 = train_and_evaluate(
    experiment_name = "type1_ordinal",
    X_train=X_tr1, y_train=y_train,
    X_test=X_te1,  y_test=y_test,
    feature_names=feats1,
)
save_experiment(result1)

---
## 4 · Experiment 2 – Binary encoding (Type 2)

In [ ]:
from encoding_strategies import encode_type2_binary

X_tr2, X_te2, feats2 = encode_type2_binary(train_raw, test_raw)

result2 = train_and_evaluate(
    experiment_name = "type2_binary",
    X_train=X_tr2, y_train=y_train,
    X_test=X_te2,  y_test=y_test,
    feature_names=feats2,
)
save_experiment(result2)

---
## 5 · Experiment 3 – Actuarial encoding (Type 3, DH-Liab / pp_bi)

In [ ]:
from encoding_strategies import encode_type3_actuarial

X_tr3, X_te3, feats3, encoders3 = encode_type3_actuarial(train_raw, test_raw)

result3 = train_and_evaluate(
    experiment_name = "type3_actuarial",
    X_train=X_tr3, y_train=y_train,
    X_test=X_te3,  y_test=y_test,
    feature_names=feats3,
)
save_experiment(result3, encoders=encoders3)

---
## 6 · Experiment 4 – Custom encoding (Type 4)

Reads `type_4_custom` column from `config/level_mapping_reference.csv`.  
Currently falls back to Type 1 until the column is populated.

In [ ]:
# from encoding_strategies import encode_type4_custom

# X_tr4, X_te4, feats4 = encode_type4_custom(train_raw, test_raw)

# result4 = train_and_evaluate(
#     experiment_name = "type4_custom",
#     X_train=X_tr4, y_train=y_train,
#     X_test=X_te4,  y_test=y_test,
#     feature_names=feats4,
# )
# save_experiment(result4)

---
## 7 · Compare all experiments

In [ ]:
from model_training import compare_results

# Add result4 here if you ran Experiment 4
all_results = [result1, result2, result3]
summary_df  = compare_results(all_results)

display(summary_df)

---
## 8 · Lift Charts – Individual Models

In [ ]:
from visualization import lift_chart, model_metrics, compare_lift_charts

# Individual lift charts for each encoding strategy
for r in all_results:
    lift_chart(r, test_df=test_raw, bins=10, print_table=True)

---
## 9 · Lift Chart Comparison – All 4 Strategies

In [ ]:
# Overlay all 4 predictions on a single lift chart + compute actuarial metrics
actuarial_summary = compare_lift_charts(all_results, test_df=test_raw, bins=10)

display(actuarial_summary)

---
## 10 · Actuarial Metrics – Per Experiment

In [ ]:
# Detailed actuarial metrics (50-bin precision)
print('Actuarial Metrics (50-bin):')
print(f'{"Experiment":<22}  {"Model Power":>12}  {"Fit Quality":>12}  {"RMSE":>10}')
print('-' * 62)
for r in all_results:
    m = model_metrics(r, test_df=test_raw, bins=50)
    print(f"{r['experiment_name']:<22}  {m['model_power']:>12.4f}  {m['fit_quality']:>12.4f}  {r['metrics']['rmse']:>10.4f}")

---
## 11 · Pipeline summary

In [ ]:
winner = summary_df["rmse"].idxmin()

print("╔══════════════════════════════════════════════════════════════╗")
print("║         XGBoost ENCODING BENCHMARK COMPLETE                  ║")
print(f"║  Mode          : {'DEBUG (10K rows)' if DEBUG else 'FULL DATA'}")
print(f"║  Target        : pp_bi")
print("╠══════════════════════════════════════════════════════════════╣")
for exp, row in summary_df.iterrows():
    flag = " ← 🏆 WINNER" if exp == winner else ""
    print(f"║  {exp:<20}  RMSE={row['rmse']:.4f}  R²={row['r2']:.4f}{flag}")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Models saved  → models/pp_bi_type*/")
print(f"║  Summary CSV   → results/encoding_benchmark_summary.csv")
print("╚══════════════════════════════════════════════════════════════╝")
print()
print("Next: provide your lift chart / SHAP code to add model comparison visuals.")